# NOTEBOOK 1 - Data Preparation & Augmentation (`tongue-dataprep`)

This notebook:
1. Loads `t2d_tongue_dataset` from Kaggle Input.
2. Prints class distribution before augmentation.
3. Applies **offline** Albumentations augmentation per class (with controlled oversampling for balance).
4. Groups photos of the same person: participant blocks (consecutive file numbers) and identity clusters (image similarity; file order only tracks participants for about the first 155 files per class).
5. `CONFIG['mode'] = 'cv'` (default): writes one augmented pool to `/kaggle/working/pool/class_name/` and `manifest.csv` with 5-fold assignments for three grouping schemes (`photo`, `block`, `cluster`); every augmentation shares its original's fold.
6. `CONFIG['mode'] = 'split'`: the earlier single `70/15/15` split by `CONFIG['split_unit']`, saved to `/kaggle/working/{train,val,test}/class_name/`.

Publish `/kaggle/working` as a Kaggle Dataset named `tongue_preped`.

In [ ]:
import os
import re
import cv2
import shutil
import random
import hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import albumentations as A
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Primary path you provided from training notebook:
DATA_ROOT = Path(os.environ.get('DATA_ROOT', '/kaggle/input/datasets/edamhakim/t2d-tongue-dataset/data/preprocessedcropped'))

OUTPUT_BASE = Path(os.environ.get('OUTPUT_BASE', '/kaggle/working'))
SPLIT_RATIOS = (0.70, 0.15, 0.15)
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

CONFIG = {
    'base_aug_per_image': 3,
    'max_aug_per_family': 6,
    'save_jpg_quality': 95,
    'run_hash_leakage_check': True,
    # Photos of one participant are numbered consecutively, so split by blocks of numbers
    'participant_block_size': 10,
    # 'block' = participant-block split (paper). 'photo' = split single photos (families) only,
    # which leaks other photos of the same participant; used for the leakage comparison run.
    'split_unit': 'block',
    # 'cv' = one pool + manifest.csv with folds for photo/block/cluster grouping (paper); 'split' = single split above
    'mode': 'cv',
    'n_folds': 5,
}
SPLITS = ('train', 'val', 'test') if CONFIG['mode'] == 'split' else ('pool',)

assert sum(SPLIT_RATIOS) == 1.0, 'Split ratios must sum to 1.0'
assert CONFIG['split_unit'] in ('block', 'photo')
assert CONFIG['mode'] in ('split', 'cv')

In [ ]:
def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTS


def canonical_key(path: Path) -> str:
    """Build a grouping key so source pre-augmented variants map to one family."""
    stem = path.stem.lower()
    stem = re.sub(r'(_?flip|_?rot(?:ate)?(?:_neg)?_?\d*|_?resized|_?aug(?:_\d+)?)', '', stem)
    stem = re.sub(r'(\(\d+\))c$', r'\1', stem)  # r_d_(19)c is a re-save of r_d_(19)
    stem = re.sub(r'__+', '_', stem).strip('_')
    return stem


def participant_key(path: Path) -> str:
    """Map r_d_(56).jpg to r_d_block005: consecutive numbers come from the same participant."""
    stem = canonical_key(path)
    m = re.search(r'\((\d+)\)', stem)
    if m is None:
        return stem
    block = (int(m.group(1)) - 1) // CONFIG['participant_block_size']
    return f'{stem[:m.start()]}block{block:03d}'


assert participant_key(Path('r_d_(1).jpg')) == participant_key(Path('r_d_(10)_flip.jpg')) == 'r_d_block000'
assert participant_key(Path('r_d_(11)c.jpg')) == 'r_d_block001'
assert canonical_key(Path('r_d_(19)c.jpg')) == canonical_key(Path('r_d_(19).jpg'))  # the release re-saves two photos with a 'c' suffix
assert participant_key(Path('r_nd_(5).jpg')) != participant_key(Path('r_d_(5).jpg'))


def collect_originals(root: Path):
    class_to_files = defaultdict(list)
    class_to_groups = defaultdict(lambda: defaultdict(list))

    # Support either root/class/*.jpg OR deeper layouts (train/val/test/class/*.jpg)
    for img_path in root.rglob('*'):
        if img_path.is_file() and is_image_file(img_path):
            class_name = img_path.parent.name.strip().lower()
            class_to_files[class_name].append(img_path)
            class_to_groups[class_name][canonical_key(img_path)].append(img_path)

    class_to_files = {k: sorted(v) for k, v in class_to_files.items() if len(v) > 0}

    if len(class_to_files) < 2:
        raise ValueError(
            f'Expected at least 2 classes, found {len(class_to_files)}. '
            'Check your dataset folder layout.'
        )

    # Pick one representative path per canonical group (prefer non-aug-looking names)
    class_to_representatives = {}
    for cls, groups in class_to_groups.items():
        reps = []
        for _, paths in groups.items():
            paths = sorted(paths)
            preferred = [p for p in paths if 'aug' not in p.stem.lower() and 'flip' not in p.stem.lower() and 'rot' not in p.stem.lower()]
            reps.append(preferred[0] if preferred else paths[0])
        class_to_representatives[cls] = sorted(reps)

    return class_to_files, class_to_representatives


def print_distribution(title: str, counts: dict):
    print(f'\n{title}')
    df = pd.DataFrame({'class': list(counts.keys()), 'count': list(counts.values())})
    print(df.sort_values('class').to_string(index=False))


def md5_file(path: Path) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

In [ ]:
# Identity clusters: presumed participants from image similarity (ported from analysis/identity_clusters.py).
# Two originals of one class are linked by the same file number, >= INLIERS SIFT+RANSAC inliers, a top-3 colour
# neighbour with Hellinger similarity >= COLOUR_TAU, or colour above the random-pair 99th percentile within 4 numbers.
# A photo still alone joins its closest colour match if similarity >= SINGLETON_TAU. Thresholds were set by eye.
INLIERS, TOP_K, NEAR, COLOUR_TAU, COLOUR_TOP, SINGLETON_TAU = 20, 10, 4, 0.85, 3, 0.70

# Eye-checked partition (analysis/identity_clusters.py, contact sheets in the paper's review folder):
# file numbers per identity cluster. These clusters define the cluster folds; the recomputed clusters
# below only check that this OpenCV build reproduces them.
REFERENCE_CLUSTERS = {
    'diabetes': [
        [10, 177, 205], [99, 100, 101, 166, 188, 225], [102, 103, 104, 226, 227], [105, 106, 107, 189, 228],
        [108, 109, 110, 229, 230, 247], [11, 12, 13, 14, 15, 20, 206], [111, 112, 113, 114, 190, 231], [115,
        116, 117, 118, 191, 232, 233], [119, 120, 121, 122, 123, 234], [124, 125, 126, 192, 235, 236], [127,
        128, 129, 193, 237], [130, 131, 132, 133, 134, 238, 239], [135, 136, 137, 138, 194, 240], [139, 140,
        141, 142, 143, 180, 195, 241], [144, 145, 146, 147, 242], [148, 149, 150, 151, 196, 243], [152, 153,
        154], [155, 244], [156, 157, 158, 197, 245], [159, 160, 161, 198, 246], [16, 17, 18, 19, 21, 207],
        [162, 163, 164, 165, 199], [78, 79, 80, 81, 82, 89, 167, 185, 220], [24, 74, 75, 76, 77, 168, 169,
        219], [66, 67, 68, 69, 70, 170, 217], [62, 63, 64, 65, 171, 183], [23, 29, 30, 31, 32, 33, 172, 179,
        209, 249], [22, 25, 26, 27, 28, 173, 178, 208], [48, 49, 50, 51, 174, 201, 214, 250], [4, 5, 175,
        203], [2, 3, 176, 202], [45, 46, 47, 181, 213], [56, 57, 58, 182], [71, 72, 73, 184, 218, 248], [83,
        84, 85, 186, 221], [91, 92, 93, 187, 224], [59, 60, 61, 200, 215, 216], [6, 7, 8, 9, 204], [34, 35,
        36, 37, 38, 210], [39, 211], [42, 43, 44, 212], [86, 87, 88, 90, 222, 223], [40, 41], [54, 55], [94,
        95, 96, 97, 98]
    ],
    'nondiabetes': [
        [1, 31, 32, 33, 34, 35, 36, 37, 38, 145, 146, 181, 185, 208, 209, 238], [10, 69, 70, 71, 72, 220],
        [97, 98, 99, 100, 226], [101, 102, 103, 104, 227], [105, 106, 107, 188], [5, 108, 109, 110, 111, 228],
        [11, 149, 150, 151, 195], [112], [113, 189], [6, 114, 115, 116, 229], [7, 117, 118, 230], [119, 120,
        121, 122, 123, 124, 125, 126, 190, 231], [12, 152, 153, 207, 239], [127, 128, 129, 191, 232], [13, 14,
        15, 16, 202], [81, 82, 130, 131, 132, 233, 234], [133, 134, 135, 136, 192], [137, 138, 139, 235],
        [140, 141, 193, 236], [22, 142, 143, 237], [8, 144, 154, 194, 240], [147], [148], [155, 196, 241],
        [156, 157, 158, 197, 242], [159, 160, 161], [9, 162, 163, 164, 198], [17, 18, 165, 166, 167, 199, 203,
        244], [168, 171, 172, 173, 174, 246], [169, 170, 245], [175], [176, 178, 247, 248, 250], [177], [179,
        201], [27, 28, 180, 206], [44, 45, 182, 211, 212], [50, 51, 183, 214], [63, 64, 184], [77, 78, 79, 80,
        186], [86, 87, 88, 187], [19, 20, 21, 204], [2, 24], [200, 249], [23, 205], [39, 40, 41, 42, 43, 46,
        47, 48, 49, 210, 213], [3, 52, 53, 54, 55, 215], [59, 60, 216, 217], [61, 62, 218], [65, 66, 67, 68,
        219], [73, 221], [83, 222], [84, 85, 223, 224], [89, 90, 225], [243], [25], [26], [29, 30], [4, 56,
        57, 58], [74, 75, 76], [91, 92, 93], [94, 95, 96]
    ],
}


def file_number(path) -> int:
    return int(re.search(r'\((\d+)\)', canonical_key(Path(path))).group(1))


def hsv_hist(path):
    hsv = cv2.cvtColor(cv2.resize(cv2.imread(str(path)), (128, 128)), cv2.COLOR_BGR2HSV)
    h = cv2.calcHist([hsv], [0, 1, 2], None, [16, 8, 8], [0, 180, 0, 256, 0, 256]).ravel()
    return np.sqrt(h / h.sum())


_sift, _bf, _kp = cv2.SIFT_create(), cv2.BFMatcher(), {}


def _features(path):
    if path not in _kp:
        _kp[path] = _sift.detectAndCompute(cv2.resize(cv2.imread(str(path), 0), (300, 300)), None)
    return _kp[path]


def sift_inliers(a, b) -> int:
    (k1, d1), (k2, d2) = _features(a), _features(b)
    if d1 is None or d2 is None or len(d1) < 2 or len(d2) < 2:
        return 0
    good = [m for m, n in (p for p in _bf.knnMatch(d1, d2, k=2) if len(p) == 2) if m.distance < 0.75 * n.distance]
    if len(good) < 8:
        return len(good)
    _, mask = cv2.findHomography(np.float32([k1[m.queryIdx].pt for m in good]),
                                 np.float32([k2[m.trainIdx].pt for m in good]), cv2.RANSAC, 5.0)
    return int(mask.sum()) if mask is not None else 0


def identity_clusters(paths, numbers):
    """Cluster id (0..k-1, in order of first appearance) for each path of one class."""
    nums = np.asarray(numbers)
    H = np.stack([hsv_hist(p) for p in paths])
    S = H @ H.T
    dn = np.abs(nums[:, None] - nums[None, :])
    p99 = np.percentile(S[dn > 10], 99)
    parent = list(range(len(paths)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    np.fill_diagonal(S, -1)
    order = np.argsort(-S, axis=1)
    for i in range(len(paths)):
        for j in set(order[i, :TOP_K]) | set(np.where(dn[i] <= NEAR)[0]):
            if j == i:
                continue
            close_colour = S[i, j] >= COLOUR_TAU and j in order[i, :COLOUR_TOP]
            if (dn[i, j] == 0 or close_colour or (S[i, j] > p99 and dn[i, j] <= NEAR)
                    or sift_inliers(*sorted((paths[i], paths[j]))) >= INLIERS):
                parent[find(i)] = find(j)
    roots = [find(i) for i in range(len(paths))]
    for i in range(len(paths)):
        if roots.count(roots[i]) == 1 and S[i, order[i, 0]] >= SINGLETON_TAU:
            parent[find(i)] = find(order[i, 0])
    roots = [find(i) for i in range(len(paths))]
    ids = {r: k for k, r in enumerate(dict.fromkeys(roots))}
    return [ids[r] for r in roots]

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'DATA_ROOT not found: {DATA_ROOT}')

dataset_root = DATA_ROOT
print(f'Dataset root: {dataset_root}')

class_to_all_files, class_to_originals = collect_originals(dataset_root)

all_counts = {cls: len(paths) for cls, paths in class_to_all_files.items()}
print_distribution('Class distribution FOUND in source path:', all_counts)

original_counts = {cls: len(paths) for cls, paths in class_to_originals.items()}
print_distribution('Class distribution BEFORE augmentation (deduplicated originals):', original_counts)

In [ ]:
# Albumentations pipeline
augmenter = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=15,
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.8,
    ),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.GaussNoise(std_range=(0.02, 0.05), p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
    ], p=0.35),
], p=1.0)


def augment_once(img_bgr: np.ndarray) -> np.ndarray:
    return augmenter(image=img_bgr)['image']


# Build family records: each representative original image = one family.
# group_id is the participant block; whole blocks go to one split to prevent leakage.
families = []
for cls, paths in class_to_originals.items():
    for i, p in enumerate(paths):
        group_id = participant_key(p) if CONFIG['split_unit'] == 'block' else canonical_key(p)
        family_id = f'{cls}_{i:06d}_{canonical_key(p)}'
        families.append({
            'family_id': family_id,
            'group_id': group_id,
            'class_name': cls,
            'src_path': p,
        })

# Augmentation policy:
# - At least CONFIG['base_aug_per_image'] augmentations per original for all classes
# - Then controlled oversampling to balance classes, with max cap per family
base_aug_per_image = CONFIG['base_aug_per_image']
base_counts = {
    cls: len(paths) * (1 + base_aug_per_image)
    for cls, paths in class_to_originals.items()
}
target_count = max(base_counts.values())

family_aug_counts = {f['family_id']: base_aug_per_image for f in families}

for cls, paths in class_to_originals.items():
    current = base_counts[cls]
    needed = target_count - current
    if needed <= 0:
        continue

    class_family_ids = [f['family_id'] for f in families if f['class_name'] == cls]
    idx = 0
    while needed > 0:
        fid = class_family_ids[idx % len(class_family_ids)]
        if family_aug_counts[fid] < CONFIG['max_aug_per_family']:
            family_aug_counts[fid] += 1
            needed -= 1
        idx += 1
        # safety break when cap blocks further balancing
        if idx > len(class_family_ids) * (CONFIG['max_aug_per_family'] + 2):
            break

augmented_counts = defaultdict(int)
for f in families:
    cls = f['class_name']
    fid = f['family_id']
    augmented_counts[cls] += 1 + family_aug_counts[fid]

print_distribution('Class distribution AFTER augmentation (planned):', dict(augmented_counts))


if CONFIG['mode'] == 'split':
    # Stratified split by participant block (70/15/15 of blocks, not of images)
    group_to_class = {f['group_id']: f['class_name'] for f in families}
    group_ids = sorted(group_to_class)
    labels = [group_to_class[g] for g in group_ids]
    if CONFIG['split_unit'] == 'block':
        assert len(group_ids) < len(families), 'participant_key grouped nothing; check the r_d_(N) file naming'

    train_groups, temp_groups, train_y, temp_y = train_test_split(
        group_ids,
        labels,
        test_size=(1 - SPLIT_RATIOS[0]),
        random_state=SEED,
        stratify=labels,
    )

    val_ratio_within_temp = SPLIT_RATIOS[1] / (SPLIT_RATIOS[1] + SPLIT_RATIOS[2])
    val_groups, test_groups, _, _ = train_test_split(
        temp_groups,
        temp_y,
        test_size=(1 - val_ratio_within_temp),
        random_state=SEED,
        stratify=temp_y,
    )

    group_split = {g: 'train' for g in train_groups}
    group_split.update({g: 'val' for g in val_groups})
    group_split.update({g: 'test' for g in test_groups})
    split_map = {f['family_id']: group_split[f['group_id']] for f in families}
    print(f'Participant blocks: train={len(train_groups)}, val={len(val_groups)}, test={len(test_groups)}')

    assert len(split_map) == len(families)

    # Ensure group leakage is impossible at plan time
    split_to_groups = defaultdict(set)
    for f in families:
        split_to_groups[split_map[f['family_id']]].add(f['group_id'])
    assert split_to_groups['train'].isdisjoint(split_to_groups['val'])
    assert split_to_groups['train'].isdisjoint(split_to_groups['test'])
    assert split_to_groups['val'].isdisjoint(split_to_groups['test'])

    split_augmented_image_counts = defaultdict(int)
    for f in families:
        fid = f['family_id']
        split_name = split_map[fid]
        split_augmented_image_counts[split_name] += family_aug_counts[fid]

    print('\nAugmented images per split (planned):')
    for s in SPLITS:
        print(f'  {s}: {split_augmented_image_counts[s]}')

In [ ]:
# Cross-validation folds (CONFIG['mode'] == 'cv'): every original is tested once per scheme; augmentations follow their family
SCHEMES = ('photo', 'block', 'cluster')
if CONFIG['mode'] == 'cv':
    for cls in class_to_originals:
        fams = [f for f in families if f['class_name'] == cls]
        nums = [file_number(f['src_path']) for f in fams]
        ref = {n: k for k, members in enumerate(REFERENCE_CLUSTERS[cls]) for n in members}
        assert sorted(ref) == sorted(nums), f'{cls}: families differ from REFERENCE_CLUSTERS'
        computed = identity_clusters([str(f['src_path']) for f in fams], nums)
        same = len(set(zip(computed, [ref[n] for n in nums]))) == len(set(computed)) == len(set(ref.values()))
        for f, n in zip(fams, nums):
            f['group_cluster'] = f'{cls}_c{ref[n]:03d}'
        sizes = np.bincount([ref[n] for n in nums])
        print(f'{cls}: {len(fams)} families -> {len(sizes)} identity clusters (largest {sizes.max()}), '
              f'{len({participant_key(f["src_path"]) for f in fams})} blocks; recomputed clusters '
              + ('match the reference' if same else f'DIFFER ({len(set(computed))} clusters), reference used'))
    for f in families:
        f['group_photo'] = f['family_id']
        f['group_block'] = participant_key(f['src_path'])
    fam_df = pd.DataFrame(families)
    fam_df['label'] = (fam_df['class_name'] == 'diabetes').astype(int)
    fam_df['number'] = fam_df['src_path'].map(file_number)
    for scheme in SCHEMES:
        fam_df[f'fold_{scheme}'] = 0
        sgkf = StratifiedGroupKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=SEED)
        for k, (_, te) in enumerate(sgkf.split(fam_df, fam_df['label'], fam_df[f'group_{scheme}'])):
            fam_df.loc[fam_df.index[te], f'fold_{scheme}'] = k

    for scheme in SCHEMES:
        g, k = f'group_{scheme}', f'fold_{scheme}'
        assert fam_df.groupby(g)[k].nunique().max() == 1, f'{scheme}: a group spans two folds'
        assert fam_df.groupby(g)['label'].nunique().max() == 1, f'{scheme}: a group spans both classes'
        assert sorted(fam_df[k].unique()) == list(range(CONFIG['n_folds'])), f'{scheme}: empty fold'
        share = fam_df.groupby(k)['label'].mean()
        assert share.between(0.30, 0.70).all(), f'{scheme}: fold class shares {share.round(2).tolist()}'
        print(f'{scheme}: originals per fold {fam_df[k].value_counts().sort_index().tolist()}, '
              f'T2D share {share.round(2).tolist()}')
    split_map = {fid: 'pool' for fid in fam_df['family_id']}

In [ ]:
def safe_stem(name: str) -> str:
    # Keep filenames portable for Kaggle dataset publishing
    allowed = set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_')
    return ''.join(ch if ch in allowed else '_' for ch in name)


# Prepare final output folders directly (no intermediate shared folder)
for split_name in SPLITS:
    split_dir = OUTPUT_BASE / split_name
    if split_dir.exists():
        shutil.rmtree(split_dir)
    split_dir.mkdir(parents=True, exist_ok=True)

for cls in class_to_originals.keys():
    for split_name in SPLITS:
        (OUTPUT_BASE / split_name / cls).mkdir(parents=True, exist_ok=True)


# Save originals + offline augmented images directly into split folders
final_counts = defaultdict(lambda: defaultdict(int))  # final_counts[split][class]
written_rows = []

for f in tqdm(families, desc='Saving augmented dataset'):
    fid = f['family_id']
    gid = f['group_id']
    cls = f['class_name']
    src = f['src_path']
    split_name = split_map[fid]

    img = cv2.imread(str(src))
    if img is None:
        print(f'[WARN] Could not read image: {src}')
        continue

    dst_dir = OUTPUT_BASE / split_name / cls

    # Save original
    orig_base = safe_stem(src.stem)
    orig_name = f'{fid}__orig__{orig_base}.jpg'
    orig_out = dst_dir / orig_name
    cv2.imwrite(str(orig_out), img, [int(cv2.IMWRITE_JPEG_QUALITY), int(CONFIG['save_jpg_quality'])])
    final_counts[split_name][cls] += 1
    written_rows.append({'filepath': str(orig_out), 'split': split_name, 'group_id': gid, 'family_id': fid, 'is_orig': 1})

    # Save offline augmentations planned for this family
    n_aug = family_aug_counts[fid]
    for j in range(n_aug):
        aug_img = augment_once(img)
        aug_name = f'{fid}__aug_{j:03d}__{orig_base}.jpg'
        aug_out = dst_dir / aug_name
        cv2.imwrite(str(aug_out), aug_img, [int(cv2.IMWRITE_JPEG_QUALITY), int(CONFIG['save_jpg_quality'])])
        final_counts[split_name][cls] += 1
        written_rows.append({'filepath': str(aug_out), 'split': split_name, 'group_id': gid, 'family_id': fid, 'is_orig': 0})


# Final report per class per split
rows = []
for split_name in SPLITS:
    for cls in sorted(class_to_originals.keys()):
        rows.append({
            'split': split_name,
            'class': cls,
            'count': final_counts[split_name][cls],
        })

final_df = pd.DataFrame(rows)
print('\nFinal image counts per class per split:')
print(final_df.to_string(index=False))

print('\nTotal images written:', int(final_df['count'].sum()))
print('Output root ready for Kaggle Dataset publish:', OUTPUT_BASE)

# Leakage checks on saved output
saved_df = pd.DataFrame(written_rows)
group_split_count = saved_df.groupby('group_id')['split'].nunique()
leaking_groups = group_split_count[group_split_count > 1]
print('Group leakage count:', int(leaking_groups.shape[0]))
assert leaking_groups.shape[0] == 0, 'Leakage detected: same group_id appears in multiple splits.'

# Optional exact-duplicate hash check across splits
if CONFIG['run_hash_leakage_check']:
    split_hashes = defaultdict(set)
    for row in written_rows:
        p = Path(row['filepath'])
        split_hashes[row['split']].add(md5_file(p))

    tv = len(split_hashes['train'].intersection(split_hashes['val']))
    tt = len(split_hashes['train'].intersection(split_hashes['test']))
    vt = len(split_hashes['val'].intersection(split_hashes['test']))

    print(f'Exact hash overlaps: train-val={tv}, train-test={tt}, val-test={vt}')
    assert tv == 0 and tt == 0 and vt == 0, 'Hash leakage detected across splits.'

# Zip archives for download (split mode; cv mode publishes pool/ + manifest.csv instead)
if CONFIG['mode'] == 'split':
    for split_name in SPLITS:
        split_dir = OUTPUT_BASE / split_name
        zip_base = OUTPUT_BASE / split_name
        shutil.make_archive(str(zip_base), 'zip', root_dir=str(split_dir))

    # Full bundle zip (train + val + test)
    bundle_dir = OUTPUT_BASE / 'tongue_preped_bundle_dir'
    if bundle_dir.exists():
        shutil.rmtree(bundle_dir)
    bundle_dir.mkdir(parents=True, exist_ok=True)
    for split_name in SPLITS:
        shutil.copytree(OUTPUT_BASE / split_name, bundle_dir / split_name)
    shutil.make_archive(str(OUTPUT_BASE / 'tongue_preped_bundle'), 'zip', root_dir=str(bundle_dir))
    shutil.rmtree(bundle_dir)
    print('\nDownloadable zip files created:')
    for name in [f'{s}.zip' for s in SPLITS] + ['tongue_preped_bundle.zip']:
        print(' -', OUTPUT_BASE / name)

legacy_csv = OUTPUT_BASE / 'metadata.csv'
if legacy_csv.exists():
    legacy_csv.unlink()
    print('Removed old file:', legacy_csv)


In [ ]:
# CV manifest: one row per written image, folds inherited from the family; exact duplicates must not cross folds
if CONFIG['mode'] == 'cv':
    cols = ['family_id', 'class_name', 'label', 'number'] + [f'{p}_{s}' for p in ('group', 'fold') for s in SCHEMES]
    manifest = pd.DataFrame(written_rows).merge(fam_df[cols], on='family_id', validate='many_to_one')
    manifest['file'] = manifest['filepath'].map(lambda p: Path(p).relative_to(OUTPUT_BASE).as_posix())
    manifest = manifest.rename(columns={'class_name': 'cls', 'family_id': 'family'})[
        ['file', 'cls', 'label', 'family', 'number', 'is_orig'] + cols[4:]]
    assert manifest['is_orig'].sum() == len(fam_df) and manifest['file'].is_unique
    md5 = manifest['file'].map(lambda f: md5_file(OUTPUT_BASE / f))
    for scheme in SCHEMES:
        crossing = md5.groupby(md5).filter(lambda s: manifest.loc[s.index, f'fold_{scheme}'].nunique() > 1)
        print(f'{scheme}: exact-duplicate files crossing folds = {crossing.nunique()}')
        assert crossing.empty, f'{scheme}: identical files in different folds'
    manifest.to_csv(OUTPUT_BASE / 'manifest.csv', index=False)
    print('Wrote', OUTPUT_BASE / 'manifest.csv', len(manifest), 'rows')

## Publish as Kaggle Dataset: `tongue_preped`

In `cv` mode the dataset is `pool/<class_name>/` plus `manifest.csv` (columns `file, cls, label, family, number, is_orig, group_photo, group_block, group_cluster, fold_photo, fold_block, fold_cluster`); the steps below describe `split` mode.

After running all cells:
1. Confirm the folders exist:
   - `/kaggle/working/train/<class_name>/`
   - `/kaggle/working/val/<class_name>/`
   - `/kaggle/working/test/<class_name>/`
2. Confirm downloadable zip files were created:
   - `/kaggle/working/train.zip`
   - `/kaggle/working/val.zip`
   - `/kaggle/working/test.zip`
   - `/kaggle/working/tongue_preped_bundle.zip`
3. Check leakage logs printed by the notebook:
   - `Group leakage count: 0`
   - `Exact hash overlaps: train-val=0, train-test=0, val-test=0`

This dataset can then be imported by your training notebook.